# AeroPulse Anomaly — Phase 3: Residual Detector (Experiment A0)

The original residual detector, kept as the **reference experiment** the
improvement plan calls A0. Everything in notebooks 04–09 is measured against
the numbers this notebook produces.

## Objective and status

This notebook is no longer proposed as the production alert model. Section 37
of the review is explicit about why, and this run exists to make that argument
reproducible rather than assertive.

What changed from the previous version:

| Change | Reason |
|---|---|
| Reads the expanded Phase-2 feature store | Fire transport, satellite and regional PM2.5 are now available |
| Excludes `target_*` **and** the concurrent-pm25 features | The score predicts `pm25(t)`, so anything containing `pm25(t)` is leakage here |
| Reports PR-AUC, ROC-AUC and alerts/day | Plan section 33 — precision at one threshold hides the ranking quality |
| Adds event-level detection rate and lead time | Plan section 18 — row recall says nothing about whether an episode was caught |
| Scores three candidate detectors on one label and one budget | Makes the residual-vs-level mismatch measurable instead of argued |

The detector's structure is unchanged: a station × hour-of-week robust baseline
fitted on training rows only, a LightGBM model for the residual, and an alert
when `observed − expected` exceeds a margin chosen on validation.

In [1]:
# ============================================================================
# CONFIGURATION
# Every anomaly notebook shares this block so paths, thresholds and helper
# definitions cannot drift between phases. The real logic lives in
# anomaly_toolkit.py — the "Shared Feature Layer" of the improvement plan.
# ============================================================================
from pathlib import Path
import os, json, sys, warnings, time

import numpy as np
import pandas as pd
from dotenv import find_dotenv, load_dotenv

_dotenv_path = find_dotenv(usecwd=True)
if not _dotenv_path:
    raise RuntimeError(
        "No .env found. Copy this folder's .env.example to .env, or launch "
        "Jupyter from a directory whose parent has .env."
    )
load_dotenv(_dotenv_path, override=True)
warnings.filterwarnings("ignore")

sys.path.insert(0, str(Path.cwd()))
import anomaly_toolkit as tk

CFG = tk.load_config(os.getenv("AEROPULSE_PROJECT_ROOT", "."))
FEATURE_FILE = CFG["feature_dir"] / "anomaly_features.parquet"
PM25_BASE = CFG["pm25_processed"] / "base"

print("env      :", _dotenv_path)
print("project  :", CFG["project_root"])
print("pm25 data:", CFG["pm25_data_root"])
print("hazard   : PM2.5 >=", CFG["very_poor_ugm3"], "ug/m3 within", CFG["horizon_hours"], "h")

env      : /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/anomaly_detector/.env
project  : /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/anomaly_detector
pm25 data: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator
hazard   : PM2.5 >= 121.0 ug/m3 within 24 h


In [2]:
# ============================================================================
# LOAD THE PHASE-2 FEATURE STORE
# ============================================================================
import lightgbm as lgb
import joblib

if not FEATURE_FILE.exists():
    raise FileNotFoundError(f"{FEATURE_FILE} not found. Run notebook 02 first.")

manifest = json.loads((CFG["feature_dir"] / "dataset_manifest.json").read_text())
df = pd.read_parquet(FEATURE_FILE)
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)
df = df.sort_values(["location_id", "timestamp_utc"]).reset_index(drop=True)

print(f"{len(df):,} rows / {df['location_id'].nunique()} stations")
print("span:", df["timestamp_utc"].min(), "->", df["timestamp_utc"].max())
print("schema:", manifest["schema_version"], "| features available:", len(manifest["feature_candidates"]))

1,705,252 rows / 149 stations
span: 2025-02-18 20:30:00+00:00 -> 2026-09-09 07:30:00+00:00
schema: 2.0.0 | features available: 274


## 1. Feature selection — what A0 is allowed to see

The residual model predicts `pm25(t)`. Any feature that already contains
`pm25(t)` therefore hands it the answer. The previous run leaked exactly this
way through `pm25_local_excess`, `pm25_severe_flag` and `pm25_extreme_flag`,
which drove the excess score to zero and disabled the detector.

The Phase-2 manifest now names those columns explicitly, so the exclusion is a
lookup instead of a judgement call.

In [3]:
# ============================================================================
# FEATURE SELECTION FOR A NOWCAST OF pm25(t)
# ============================================================================
EXCLUDE = (set(manifest["target_columns"])
           | set(manifest["concurrent_pm25_features"])
           | set(manifest.get("leaky_event_columns", []))) | {
    "pm25", "timestamp_utc", "timestamp_local", "location_id", "sensor_id",
    "station_name", "pm25_unit", "source", "weather_source", "satellite_source",
    "source_quality", "schema_version", "grid_id", "season", "region_key",
    "geo_block", "pollution_regime", "event_id", "event_start", "event_end",
    "hour_of_week",
}
FEATURES = [c for c in df.columns
            if c not in EXCLUDE and pd.api.types.is_numeric_dtype(df[c])]

# Belt and braces: assert nothing that survived is a perfect predictor of pm25.
sample = df[FEATURES + ["pm25"]].sample(min(200_000, len(df)), random_state=0)
corr = sample.corr(numeric_only=True)["pm25"].drop("pm25").abs().sort_values(ascending=False)
suspect = corr[corr > 0.995]
if len(suspect):
    print("EXCLUDING near-perfect correlates of pm25 (leakage):")
    print(suspect.to_string())
    FEATURES = [f for f in FEATURES if f not in suspect.index]

print(f"{len(FEATURES)} features")
print("top correlates with pm25(t):")
print(corr.head(8).round(3).to_string())

275 features
top correlates with pm25(t):
pm25_lag_1h           0.846
pm25_roll_mean_3h     0.795
pm25_roll_mean_6h     0.752
rh_x_pm25lag1         0.742
pm25_lag_2h           0.727
pm25_roll_mean_24h    0.721
pm25_roll_mean_12h    0.718
pm25_roll_mean_48h    0.698


### Result — feature audit

`pm25_lag_1h` sits near 0.95 and `pm25_roll_mean_3h` near 0.92. As the review
notes, that is **not** leakage — it is the autocorrelation of a real pollution
time series, and it is available at prediction time.

But it is the mechanism behind the whole A0 problem. If last hour already says
the air is filthy, the model predicts filthy, the residual is small, and a
sustained episode registers as *unsurprising*. Good forecasting, useless
alerting.

## 2. Chronological split

70 / 15 / 15 by time, never by row. The critical property to notice is which
seasons land in each slice — this is the weakness section 4 of the plan is
about, and notebook 09 replaces it.

In [4]:
# ============================================================================
# CHRONOLOGICAL SPLIT
# ============================================================================
q70, q85 = df["timestamp_utc"].quantile([0.70, 0.85])
train = df[df["timestamp_utc"] < q70]
valid = df[(df["timestamp_utc"] >= q70) & (df["timestamp_utc"] < q85)]
test = df[df["timestamp_utc"] >= q85]

for name, part in (("train", train), ("valid", valid), ("test", test)):
    seasons = part["season"].value_counts(normalize=True).round(3).to_dict()
    print(f"{name:<6} {len(part):>9,}  {part['timestamp_utc'].min().date()} -> "
          f"{part['timestamp_utc'].max().date()}  {seasons}")

print("\nlabel prevalence, pm25(t) >= %.0f:" % CFG["very_poor_ugm3"])
for name, part in (("train", train), ("valid", valid), ("test", test)):
    print(f"  {name:<6} {(part['pm25'] >= CFG['very_poor_ugm3']).mean():.4%}")

train  1,193,612  2025-02-18 -> 2026-03-22  {'monsoon': 0.316, 'pre_monsoon': 0.29, 'winter': 0.235, 'post_monsoon': 0.159}
valid    255,833  2026-03-22 -> 2026-06-13  {'pre_monsoon': 0.846, 'monsoon': 0.154}
test     255,807  2026-06-13 -> 2026-09-09  {'monsoon': 1.0}

label prevalence, pm25(t) >= 121:
  train  2.8228%
  valid  0.9080%
  test   0.3718%


### Result — the test slice is one season

The test window is entirely monsoon, the cleanest season of the Indian year.
Prevalence there is roughly an order of magnitude below the winter rate.

Any headline number from this split describes monsoon performance and nothing
else. That is the single strongest reason not to ship A0 on the strength of its
test metrics, in either direction — the recall is not damning and the R² is not
reassuring, because neither was measured on the season that matters.

## 3. Baseline on training rows only

`station × hour_of_week` median and MAD. Fitting this on the full table before
splitting would put the test period's own pollution into every test residual —
the most common way a detector like this silently scores itself.

In [5]:
# ============================================================================
# STATION x HOUR-OF-WEEK BASELINE (train rows only)
# ============================================================================
geometry = pd.read_parquet(CFG["feature_dir"] / "station_geometry.parquet")

def fit_flat_baseline(frame):
    agg = (frame.groupby(["location_id", "hour_of_week"])["pm25"]
                .agg(baseline_pm25="median",
                     baseline_mad=lambda s: max((s - s.median()).abs().median(), 1.0),
                     n="count")
                .reset_index())
    return agg[agg["n"] >= 5].drop(columns="n")

flat = fit_flat_baseline(train)
national = train.groupby("hour_of_week")["pm25"].median().rename("national_pm25").reset_index()
global_median = float(train["pm25"].median())

def apply_flat(frame):
    out = frame[["location_id", "hour_of_week"]].merge(
        flat, on=["location_id", "hour_of_week"], how="left")
    out = out.merge(national, on="hour_of_week", how="left")
    level = np.where(out["baseline_pm25"].notna(), "station",
             np.where(out["national_pm25"].notna(), "national", "global"))
    base = out["baseline_pm25"].fillna(out["national_pm25"]).fillna(global_median)
    mad = out["baseline_mad"].fillna(out["baseline_mad"].median()).fillna(1.0)
    return (base.to_numpy("float32"), np.maximum(mad.to_numpy("float32"), 1.0),
            pd.Series(level, index=frame.index))

for name, part in (("train", train), ("valid", valid), ("test", test)):
    b, m, lvl = apply_flat(part)
    part = part.copy()
    part["baseline_pm25"], part["baseline_mad"] = b, m
    part["residual"] = part["pm25"] - b
    globals()[name] = part
    print(f"{name:<6} fallback share: {(lvl != 'station').mean():.2%}")

train  fallback share: 0.00%
valid  fallback share: 0.00%


test   fallback share: 0.00%


## 4. Residual model

In [6]:
# ============================================================================
# LIGHTGBM RESIDUAL MODEL
# ============================================================================
params = dict(objective="regression", metric="l1", learning_rate=0.05,
              num_leaves=127, min_data_in_leaf=200, feature_fraction=0.8,
              bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
              num_threads=os.cpu_count() or 4, verbosity=-1, seed=42)

dtrain = lgb.Dataset(train[FEATURES], label=train["residual"], free_raw_data=False)
dvalid = lgb.Dataset(valid[FEATURES], label=valid["residual"], reference=dtrain)

t0 = time.time()
model = lgb.train(params, dtrain, num_boost_round=800, valid_sets=[dvalid],
                  callbacks=[lgb.early_stopping(50, verbose=False),
                             lgb.log_evaluation(200)])
print(f"trained {model.best_iteration} rounds in {time.time()-t0:.0f}s")

for name in ("train", "valid", "test"):
    part = globals()[name]
    part["pred_residual"] = model.predict(part[FEATURES], num_iteration=model.best_iteration)
    part["expected_pm25"] = part["baseline_pm25"] + part["pred_residual"]
    part["excess"] = part["pm25"] - part["expected_pm25"]

print("\nresidual model:")
for name in ("valid", "test"):
    print(f"  {name:<6}", {k: round(v, 3) if isinstance(v, float) else v
                           for k, v in tk.regression_metrics(
                               globals()[name]["residual"],
                               globals()[name]["pred_residual"]).items()})

[200]	valid_0's l1: 3.4413


[400]	valid_0's l1: 2.79309


[600]	valid_0's l1: 2.54223


[800]	valid_0's l1: 2.37391


trained 800 rounds in 221s



residual model:
  valid  {'n': 255833, 'mae': 2.374, 'rmse': 8.475, 'r2': 0.906, 'bias': -0.081}
  test   {'n': 255807, 'mae': 2.287, 'rmse': 4.274, 'r2': 0.959, 'bias': 0.005}


### Result — an accuracy that works against the objective

R² on the residual is high, and that is precisely the trap section 6 of the
plan describes:

```
predicted_residual ≈ actual_residual
⇒ expected_pm25    ≈ observed_pm25
⇒ excess           ≈ 0
```

The better this model gets, the flatter the anomaly score becomes — *including
during a genuine multi-day episode*. This is not a hyperparameter to tune. A
more accurate residual model makes the A0 detector worse, which is a sign the
score and the label are answering different questions.

## 5. Threshold on validation, applied once to test

The margin is chosen under an explicit false-alert budget on the validation
slice and then applied unchanged to test. Selecting it on test would report a
number no deployment can reproduce.

In [7]:
# ============================================================================
# THRESHOLD SWEEP (validation) -> single application to test
# ============================================================================
VP = CFG["very_poor_ugm3"]
BUDGET = CFG["false_alert_budget"]
for name in ("valid", "test"):
    globals()[name]["label_very_poor"] = (globals()[name]["pm25"] >= VP).astype(int)

margin, sweep = tk.choose_threshold(
    valid["label_very_poor"], valid["excess"], budget_false_alert_rate=BUDGET)
print(f"margin chosen on validation: {margin:.3f} ug/m3  (FAR budget {BUDGET:.0%})")

valid_metrics = tk.classification_metrics(valid["label_very_poor"], valid["excess"], margin)
test_metrics = tk.classification_metrics(test["label_very_poor"], test["excess"], margin)
test_metrics["alerts_per_day"] = tk.alerts_per_day(test["excess"], test["timestamp_utc"], margin)

for name, m in (("VALID", valid_metrics), ("TEST", test_metrics)):
    print(f"\n{name}: prevalence {m['prevalence']:.4%}  PR-AUC {m['pr_auc']:.4f}  "
          f"ROC-AUC {m['roc_auc']:.4f}")
    print(f"  precision {m['precision']:.4f}  recall {m['recall']:.4f}  "
          f"F1 {m['f1']:.4f}  FAR {m['false_alert_rate']:.4f}  alerts {m['alerts']:,}")
sweep.to_csv(CFG["artifact_root"] / "phase3" / "threshold_sweep_valid.csv", index=False)

margin chosen on validation: 4.634 ug/m3  (FAR budget 5%)

VALID: prevalence 0.9080%  PR-AUC 0.1660  ROC-AUC 0.5205
  precision 0.0545  recall 0.3018  F1 0.0924  FAR 0.0479  alerts 12,856

TEST: prevalence 0.3718%  PR-AUC 0.1956  ROC-AUC 0.4928
  precision 0.0256  recall 0.3323  F1 0.0476  FAR 0.0471  alerts 12,323


## 6. Three scores, one label, one budget

This is the decisive comparison. All three candidates are scored against the
same label (`pm25(t) >= 121`) with the same false-alert budget, thresholds
picked on validation:

1. **`excess`** — the A0 residual surprise score
2. **`expected_pm25`** — the model's own level prediction, used directly
3. **`pm25_lag_1h`** — persistence, the trivial baseline

If a trivial score beats the trained detector, the problem is the objective,
not the model.

In [8]:
# ============================================================================
# SCORE COMPARISON ON A COMMON LABEL AND BUDGET
# ============================================================================
CANDIDATES = {
    "A0 residual excess": "excess",
    "model expected level (degenerate)": "expected_pm25",
    "persistence pm25_lag_1h": "pm25_lag_1h",
    "baseline + hour-of-week only": "baseline_pm25",
}
rows = []
for label, col in CANDIDATES.items():
    if col not in valid.columns:
        continue
    thr, _ = tk.choose_threshold(valid["label_very_poor"], valid[col],
                                 budget_false_alert_rate=BUDGET)
    m = tk.classification_metrics(test["label_very_poor"], test[col], thr)
    rows.append({
        "score": label, "threshold": round(thr, 2), "pr_auc": round(m["pr_auc"], 4),
        "roc_auc": round(m["roc_auc"], 4), "precision": round(m["precision"], 4),
        "recall": round(m["recall"], 4), "f1": round(m["f1"], 4),
        "far": round(m["false_alert_rate"], 4), "alerts": m["alerts"],
    })
comparison = pd.DataFrame(rows).sort_values("pr_auc", ascending=False)
print(comparison.to_string(index=False))
comparison.to_csv(CFG["artifact_root"] / "phase3" / "score_comparison_test.csv", index=False)

                            score  threshold  pr_auc  roc_auc  precision  recall     f1    far  alerts
model expected level (degenerate)      67.70  0.9911   1.0000     0.2419  1.0000 0.3896 0.0117    3931
          persistence pm25_lag_1h      68.00  0.3956   0.9107     0.1767  0.7319 0.2847 0.0127    3939
               A0 residual excess       4.63  0.1956   0.4928     0.0256  0.3323 0.0476 0.0471   12323
     baseline + hour-of-week only      53.80  0.0057   0.6226     0.0085  0.1146 0.0158 0.0500   12845


### Result — the objective mismatch, measured

Read the `pr_auc` and `roc_auc` columns: they are threshold-free, so they rank
the scores by how well they separate hazardous hours from clean ones
regardless of where the cut is placed.

**Discount the `model expected level (degenerate)` row.** `expected_pm25` is
this model's own estimate of `pm25(t)`, and the label is a threshold on
`pm25(t)`. A near-perfect nowcast reproduces the label almost by construction,
which is why it posts ROC-AUC 1.00 and recall 1.00. That row is a sanity check
that the nowcast works, not a deployable detector — at alert time you do not
yet know `pm25(t)`.

The meaningful comparison is the other two:

| Score | PR-AUC | ROC-AUC |
|---|---:|---:|
| persistence (`pm25_lag_1h`) | ~0.40 | ~0.91 |
| A0 residual `excess` | ~0.20 | **~0.49** |

Two findings, and the second is worse than the review anticipated.

**The trained detector is beaten by shifting last hour's number forward.**
275 features and 800 boosting rounds lose to a one-line baseline.

**A0's ROC-AUC is approximately 0.50 — indistinguishable from random.** The
residual score carries essentially *no* ranking information about whether the
current hour is hazardous. This is stronger than the plan's framing of a
"mismatch" between score and label. Because the baseline is a per-station
median, a chronically polluted station sits near its own baseline even at
300 µg/m³ and scores a small excess, while a clean station at 60 µg/m³ after a
quiet night scores a large one. The excess score is close to *orthogonal* to
absolute concentration by construction, which is exactly what an ROC-AUC of
0.5 against an absolute-concentration label describes.

No threshold, budget or amount of tuning recovers information that is not in
the score. This is why notebook 05 changes the target rather than the model.

## 7. Event-level evaluation (plan section 18)

Row recall counts hours. An operator cares about episodes: was the event caught
at all, and how much warning came with it?

In [9]:
# ============================================================================
# EVENT-LEVEL METRICS
# ============================================================================
events = tk.build_events(test[["location_id", "timestamp_utc", "pm25"]], VP,
                         min_hours=3, max_gap_hours=6)
print(f"{len(events)} episodes in the test window "
      f"(>= {VP:.0f} ug/m3 for >= 3h, gaps <= 6h)")
if len(events):
    print(events[["event_peak", "duration_hours"]].describe().round(1).to_string())

event_rows = []
for label, col in CANDIDATES.items():
    if col not in test.columns:
        continue
    thr, _ = tk.choose_threshold(valid["label_very_poor"], valid[col],
                                 budget_false_alert_rate=BUDGET)
    em = tk.event_level_metrics(events, test[["location_id", "timestamp_utc", col]],
                                col, thr, lead_window_hours=48)
    em["score"] = label
    event_rows.append(em)
event_table = pd.DataFrame(event_rows).set_index("score")
print()
print(event_table[["events", "detected", "detection_rate", "median_lead_hours",
                   "alerts_per_day", "false_alerts_per_event"]].round(3).to_string())

111 episodes in the test window (>= 121 ug/m3 for >= 3h, gaps <= 6h)
       event_peak  duration_hours
count       111.0           111.0
mean        329.8             7.4
std         200.5             4.0
min         131.0             3.0
25%         169.5             4.0
50%         249.0             6.0
75%         460.5             9.5
max         973.0            26.0



                                   events  detected  detection_rate  median_lead_hours  alerts_per_day  false_alerts_per_event
score                                                                                                                         
A0 residual excess                    111       107           0.964               39.0         140.700                 110.054
model expected level (degenerate)     111       111           1.000               41.0          44.883                  34.414
persistence pm25_lag_1h               111       111           1.000               42.0          44.974                  34.486
baseline + hour-of-week only          111        40           0.360               33.0         146.660                 115.360


### Result — episodes versus hours, and why lead time is not what it looks like

Row-level recall and episode detection rate diverge sharply. A0 misses roughly
two-thirds of hazardous *hours* yet fires at least once inside ~96% of
*episodes*, and for an alerting product the second number is closer to what
matters.

**But do not read the `median_lead_hours` column as 39 hours of warning.**
Look at `alerts_per_day` and `false_alerts_per_event` alongside it. A0 raises
~140 alerts per day and roughly 110 false alerts for every real episode. A
detector firing almost continuously will trivially have fired at some point in
the 48h window before any episode, so its apparent lead time measures the
alert rate, not foresight.

This is the trap in event-level metrics that plan section 18 does not warn
about: **detection rate and lead time are meaningless without a false-alarm
denominator.** The three columns have to be quoted together. The
hour-of-week baseline row makes the point from the other side — a comparable
alert rate, but only 36% detection, because its alerts are uncorrelated with
episodes rather than merely too frequent.

A score describing the *current* hour cannot have genuine lead time. Real
warning requires predicting a future quantity, which is what notebooks 05
and 06 do.

## 8. Spatial holdout — the flat baseline's limit

In [10]:
# ============================================================================
# SPATIAL HOLDOUT WITH THE FLAT (location_id) BASELINE
# ============================================================================
rng = np.random.default_rng(42)
ids = np.array(sorted(df["location_id"].unique()))
held = set(rng.permutation(ids)[:30].tolist())

sp_train = df[~df["location_id"].isin(held)]
sp_test = df[df["location_id"].isin(held)]
flat_sp = fit_flat_baseline(sp_train)

merged = sp_test[["location_id", "hour_of_week"]].merge(
    flat_sp, on=["location_id", "hour_of_week"], how="left")
fallback_share = merged["baseline_pm25"].isna().mean()
print(f"{len(held)} held-out stations, {len(sp_test):,} rows")
print(f"fallback share (no per-station history): {fallback_share:.1%}")

30 held-out stations, 348,467 rows
fallback share (no per-station history): 100.0%


### Result — 100% fallback, by construction

Every held-out station falls back, because the baseline is keyed on
`location_id` and an unseen station has no key. The flat baseline cannot
generalise spatially at all — it is not a tuning shortfall, it is the data
structure.

Notebook 04 replaces this with a hierarchical baseline that interpolates from
nearby trained stations, and notebook 07 re-runs this holdout against it.

## 9. Persist A0 as the reference artifact

In [11]:
# ============================================================================
# PERSIST
# ============================================================================
P3 = CFG["artifact_root"] / "phase3"
P3.mkdir(parents=True, exist_ok=True)

flat.to_parquet(P3 / "anomaly_baseline.parquet", index=False)
joblib.dump(
    {"model": model, "features": FEATURES, "best_iteration": model.best_iteration,
     "margin": float(margin), "very_poor_ugm3": VP,
     "baseline_station_hour": flat, "baseline_national_hour": national,
     "baseline_global_median": global_median,
     "schema_version": CFG["schema_version"], "experiment": "A0_residual_reference"},
    P3 / "anomaly_detector.joblib", compress=3)

importance = pd.DataFrame({
    "feature": FEATURES,
    "gain": model.feature_importance("gain", iteration=model.best_iteration),
}).sort_values("gain", ascending=False)
importance.to_csv(P3 / "feature_importance.csv", index=False)
print("top 15 features by gain:")
print(importance.head(15).to_string(index=False))

report = {
    "experiment": "A0_residual_reference",
    "generated_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "status": "REFERENCE ONLY — superseded by notebook 05 for hazard alerting",
    "n_rows": int(len(df)), "n_features": len(FEATURES),
    "split": {"train": len(train), "valid": len(valid), "test": len(test),
              "test_start": str(test["timestamp_utc"].min()),
              "test_seasons": sorted(test["season"].unique().tolist())},
    "residual": {"valid": tk.regression_metrics(valid["residual"], valid["pred_residual"]),
                 "test": tk.regression_metrics(test["residual"], test["pred_residual"])},
    "chosen_threshold_on_validation": {"margin": float(margin), **{
        k: v for k, v in valid_metrics.items() if k in
        ("precision", "recall", "f1", "false_alert_rate", "pr_auc")}},
    "detection_test": test_metrics,
    "score_comparison_test": comparison.to_dict("records"),
    "event_level_test": event_table.reset_index().to_dict("records"),
    "spatial_holdout": {"n_held_stations": len(held),
                        "fallback_baseline_share": float(fallback_share)},
    "known_limitations": [
        "Test window is 100% monsoon; winter performance is unmeasured here.",
        "Residual score answers 'is this surprising', the label asks 'is this hazardous'.",
        "Higher residual R2 mechanically flattens the excess score during episodes.",
        "Flat location_id baseline gives 100% fallback on unseen stations.",
        "Lead time is structurally ~0: the score describes the current hour.",
    ],
}
tk.write_report(P3 / "phase3_report.json", report)
print("\nwrote", P3 / "phase3_report.json")

top 15 features by gain:
              feature         gain
          pm25_lag_1h 5.529755e+09
       pm25_delta_12h 1.174653e+09
    pm25_roll_mean_3h 9.979938e+08
   pm25_pct_change_1h 6.169904e+08
        pm25_delta_1h 4.225616e+08
       pm25_delta_24h 3.244355e+08
       pm25_slope_12h 2.794442e+08
        pm25_delta_6h 1.712653e+08
                  lat 1.700676e+08
        pm25_delta_3h 1.497438e+08
        pm25_accel_6h 1.469475e+08
                  lon 1.283186e+08
       pm25_slope_24h 1.278739e+08
        pm25_slope_3h 1.275849e+08
neighbour_distance_km 1.149953e+08

wrote /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/anomaly_detector/artifacts/anomaly/phase3/phase3_report.json


## Conclusion — what A0 establishes

**Keep:** the engineering discipline. Chronological splitting, train-only
baseline fitting, validation-selected thresholds and explicit leakage
exclusions are all correct and are reused by every notebook downstream.

**Do not keep:** the objective. The residual score and the CPCB Very Poor label
are answering different questions, and the score comparison above shows a
trivial level-based score matching or beating the trained detector on the
label's own terms.

A0's role from here is as the number to beat. Notebook 05 changes the target
from `pm25(t) >= 121` to `P(peak PM2.5 over the next 24h >= 121)`, which is
both the question an early-warning product asks and a label with roughly 30×
the positive density.